# Código de concatenação
*by Miguel Ferreira*
Agora, criaremos um algoritmo que concatenará todos os datasets gerados pelas ferramentas do envelope de risco.

## 1. Importação

In [1]:
import pandas as pd
import numpy as np

epf  = pd.read_parquet("C:/projects/Libellula/data/processed/epf/epf_features.parquet")
qf   = pd.read_parquet("C:/projects/Libellula/data/processed/qf/qf_features.parquet")
cvar = pd.read_parquet("C:/projects/Libellula/data/processed/cvar/cvar_features.parquet")
pdfd = pd.read_parquet("C:/projects/Libellula/data/processed/pdfd/pdfd_features.parquet")
evt  = pd.read_parquet("C:/projects/Libellula/data/processed/evt/evt_features.parquet")


## 2. Teste de alinhamento

In [2]:
dfs = [epf, qf, cvar, pdfd, evt]

base = dfs[0].index

for i, df in enumerate(dfs):
    assert df.index.equals(base), f"Dataset {i} desalinhado"

Nenhum desalinhamento encontrado.

In [3]:
print(len(epf), len(qf), len(cvar), len(pdfd), len(evt))

205 205 205 205 205


Todos os datasets estão do mesmo tamanho. Agora transformamos todas as colunas de todos os datasets em tipo ```gloat64```

In [4]:
for df in [epf, qf, cvar, pdfd, evt]:
    for col in df.columns:
        if df[col].dtype != "float32":
            df[col] = df[col].astype("float32")

In [5]:
for df in [epf, qf, cvar, pdfd, evt]:
    df.index.name = "timestamp"

In [6]:
for df in [epf, qf, cvar, pdfd, evt]:
    assert df.index.name == "timestamp"
    assert df.index.is_monotonic_increasing
    assert not df.index.has_duplicates
    assert all(df.dtypes == "float32")

## 3. Enfim, a concatenação!!!

In [7]:
risk = pd.concat([epf, qf, cvar, pdfd, evt], axis=1)
risk = risk.dropna()

risk = risk.astype("float32")
risk.index.name = "timestamp"

risk

,epf_upper,epf_lower,epf_width,epf_asymmetry,quantile_lower,q25,q50,q75,quantile_upper,quantile_width,...,pdfd_005,pdfd_01,pdfd_02,pdfd_03,pdf_skew,pdf_kurtosis,evt_shape,evt_scale,evt_var,evt_cvar
timestamp,,,,,,,,,,,,,,,,,,,,,
2024-07-18,1.648634,0.530566,1.118068,0.000000e+00,-0.006797,-0.002225,-0.000400,0.002021,0.006149,0.012946,...,0.095238,0.003968,0.0,0.0,0.123299,1.574841,1.393293,0.000457,0.205143,-0.505447
2024-07-19,1.663191,0.512209,1.150982,0.000000e+00,-0.006797,-0.002202,-0.000079,0.001899,0.006149,0.012946,...,0.095238,0.003968,0.0,0.0,0.121387,1.605461,1.393293,0.000457,0.205143,-0.505447
2024-07-22,1.653558,0.524242,1.129315,-1.966188e-16,-0.006797,-0.002277,-0.000125,0.002120,0.006149,0.012946,...,0.095238,0.003968,0.0,0.0,0.120232,1.603456,1.393293,0.000457,0.205143,-0.505447
2024-07-23,1.648393,0.521807,1.126585,0.000000e+00,-0.006758,-0.002260,-0.000174,0.002118,0.006149,0.012906,...,0.095238,0.003968,0.0,0.0,0.131729,1.649436,1.393293,0.000457,0.205143,-0.505447
2024-07-24,1.644851,0.522949,1.121902,0.000000e+00,-0.006758,-0.002685,-0.000176,0.002007,0.006149,0.012906,...,0.095238,0.003968,0.0,0.0,0.132848,1.647906,1.393293,0.000457,0.205143,-0.505447
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2025-04-24,1.687553,0.590247,1.097307,0.000000e+00,-0.006618,-0.003052,-0.000537,0.002935,0.006888,0.013507,...,0.091270,0.007937,0.0,0.0,0.719145,3.030521,1.507101,0.000991,0.692114,-1.327573
2025-04-25,1.680325,0.592675,1.087649,0.000000e+00,-0.006618,-0.001962,0.000188,0.002762,0.006262,0.012880,...,0.091270,0.007937,0.0,0.0,0.721457,3.024573,1.507101,0.000991,0.692114,-1.327573
2025-04-28,1.684615,0.599785,1.084829,0.000000e+00,-0.006508,-0.003019,-0.000137,0.002562,0.006149,0.012657,...,0.091270,0.007937,0.0,0.0,0.717714,2.996909,1.507101,0.000991,0.692114,-1.327573


In [8]:
print(risk.shape)
print(risk.dtypes.unique())

(205, 25)
[dtype('float32')]


## 4. Salvamento

In [9]:
risk.to_parquet(
    "C:/projects/Libellula/data/processed/combined/risk_features.parquet"
)

risk_diagnostics = pd.DataFrame({
    "missing_values": risk.isna().sum(),
    "variance": risk.var(),
    "unique_values": risk.nunique(),
})
high_correlation_pairs = (
    risk.corr().abs().where(np.triu(np.ones((risk.shape[1], risk.shape[1])), k=1).astype(bool)).stack().sort_values(ascending=False)
)
print(risk_diagnostics)
print(high_correlation_pairs.head(10))

                       missing_values      variance  unique_values
epf_upper                           0  7.678962e-04            205
epf_lower                           0  3.878706e-03            205
epf_width                           0  5.342125e-03            205
epf_asymmetry                       0  1.677583e-33             10
quantile_lower                      0  1.645054e-06             72
q25                                 0  1.188908e-06            199
q50                                 0  1.307483e-06            197
q75                                 0  8.103631e-07            195
quantile_upper                      0  9.188565e-07             19
quantile_width                      0  4.001367e-06             92
quantile_skew                       0  2.196948e-06            205
asymmetry                           0  1.960222e-02            197
cvar_95                             0  3.917859e-06             26
cvar_99                             0  2.130104e-05           